# Weather Data Transformation

This notebook reads the raw weather JSON files from the API, flattens the records into a usable DataFrame, and saves them to SQLite.

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

raw_dir = Path('data/raw')
db_path = 'data/processed/weather.db'

current_rows = []
daily_rows = []

for file in sorted(raw_dir.glob('*.json')):
    with file.open('r', encoding='utf-8') as f:
        payload = json.load(f)

    city_name = file.stem.replace('_current', '').replace('_forecast', '').replace('_historical', '').replace('_', ' ').title()

    if 'current' in file.name:
        current = payload.get('current', {})
        current_rows.append({
            'city': city_name,
            'source': 'current',
            'timestamp': current.get('time'),
            'date': current.get('time', '')[:10],
            'temperature_2m_c': current.get('temperature_2m'),
            'relative_humidity_2m_pct': current.get('relative_humidity_2m'),
            'apparent_temperature_c': current.get('apparent_temperature'),
            'precipitation_mm': current.get('precipitation'),
            'weather_code': current.get('weather_code'),
            'wind_speed_10m_kmh': current.get('wind_speed_10m'),
        })

    elif 'forecast' in file.name or 'historical' in file.name:
        daily = payload.get('daily', {})
        times = daily.get('time', [])
        for i, date_key in enumerate(times):
            daily_rows.append({
                'city': city_name,
                'source': 'forecast' if 'forecast' in file.name else 'historical',
                'date': date_key,
                'temperature_2m_max_c': daily.get('temperature_2m_max', [None] * len(times))[i],
                'temperature_2m_min_c': daily.get('temperature_2m_min', [None] * len(times))[i],
                'precipitation_sum_mm': daily.get('precipitation_sum', [None] * len(times))[i],
                'relative_humidity_2m_mean_pct': daily.get('relative_humidity_2m_mean', [None] * len(times))[i],
                'wind_speed_10m_max_kmh': daily.get('wind_speed_10m_max', [None] * len(times))[i],
                'precipitation_probability_max_pct': daily.get('precipitation_probability_max', [None] * len(times))[i],
            })

current_df = pd.DataFrame(current_rows)
daily_df = pd.DataFrame(daily_rows)

print('Current rows:')
print(current_df)
print('\nDaily rows head:')
print(daily_df.head())

conn = sqlite3.connect(db_path)
current_df.to_sql('weather_current', conn, index=False, if_exists='replace')
daily_df.to_sql('weather_daily', conn, index=False, if_exists='replace')
conn.commit()
conn.close()

print('\nSaved to SQLite:', db_path)